In [23]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import seaborn as sns
import plot_network_maps
from _helpers import mock_snakemake
from cartopy import crs as ccrs
import time
import shapely.geometry
from tqdm import tqdm
# from plot_network_maps import plot_demand_map, plot_capacity_map, plot_base_capacity_map

In [2]:
input_network = "../resources/SoCal/western/elec_s_call_ec_lv1.0_REM-3h_E.nc"
n = pypsa.Network(input_network)
# Plot demand_map

INFO:pypsa.io:Imported network elec_s_call_ec_lv1.0_REM-3h_E.nc has buses, carriers, generators, global_constraints, lines, loads, storage_units


In [3]:
start_time = time.time()
n.lpf(snapshots=n.snapshots[:1])
print("Power flow calculation took %s seconds ---" % (time.time() - start_time))

INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 0 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')
INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 1 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')
INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 2 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')
INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 3 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')
INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 4 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')


Power flow calculation took 50.53920125961304 seconds ---


In [4]:
buses_gis = n.buses.copy()

In [5]:
buses_gis.columns

Index(['Pd', 'v_nom', 'country', 'county', 'reeds_zone', 'reeds_ba',
       'interconnect', 'x', 'y', 'nerc_reg', 'trans_reg', 'trans_grp',
       'reeds_state', 'rec_trading_zone', 'control', 'generator',
       'substation_lv', 'type', 'carrier', 'unit', 'v_mag_pu_set',
       'v_mag_pu_min', 'v_mag_pu_max', 'sub_network'],
      dtype='object')

In [6]:
n.transformers

attribute,bus0,bus1,type,model,x,r,g,b,s_nom,s_nom_mod,...,v_ang_min,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt
Transformer,,,,,,,,,,,,,,,,,,,,,


In [7]:
buses_gis = n.buses.copy()
buses_gis["geometry"] = [shapely.geometry.Point(xy) for xy in zip(buses_gis["x"], buses_gis["y"])]
gdf_buses = gpd.GeoDataFrame(buses_gis, geometry="geometry", crs="EPSG:4326")  # WGS84 lat/lon
gdf_buses.to_file(f"../results/SoCal/western/buses_orig.geojson", driver="GeoJSON")

lines = n.lines.copy()
bus_coords = n.buses[["x", "y"]]
lines["x0"] = lines["bus0"].map(bus_coords["x"])
lines["y0"] = lines["bus0"].map(bus_coords["y"])
lines["x1"] = lines["bus1"].map(bus_coords["x"])
lines["y1"] = lines["bus1"].map(bus_coords["y"])
lines["geometry"] = [
    shapely.geometry.LineString([(x0, y0), (x1, y1)])
    for x0, y0, x1, y1 in zip(lines["x0"], lines["y0"], lines["x1"], lines["y1"])
]
gdf_lines = gpd.GeoDataFrame(lines, geometry="geometry", crs="EPSG:4326")
gdf_lines.to_file(f"../results/SoCal/western/lines_orig.geojson", driver="GeoJSON")

generators = n.generators.copy()
generators["x"] = generators["bus"].map(bus_coords["x"])
generators["y"] = generators["bus"].map(bus_coords["y"])
generators["geometry"] = [shapely.geometry.Point(xy) for xy in zip(generators["x"], generators["y"])]
gdf_generators = gpd.GeoDataFrame(generators, geometry="geometry", crs="EPSG:4326")
gdf_generators.to_file(f"../results/SoCal/western/generators_orig.geojson", driver="GeoJSON")

/resnick/groups/enceladus/jyzhao/pypsa-usa/pypsa-usa/lib/python3.11/site-packages/pyogrio/raw.py:530: RuntimeWarning:

NaN of Infinity value found. Skipped

INFO:pyogrio._io:Created 1,096 records
INFO:pyogrio._io:Created 1,564 records
INFO:pyogrio._io:Created 5,446 records


In [8]:
for i, row in n.sub_networks.iterrows():
    print(i, row.obj.buses().shape)
print(n.generators.shape)

0 (1088, 24)
1 (1, 24)
2 (5, 24)
3 (1, 24)
4 (1, 24)
(5446, 38)


In [14]:
print(n.buses.shape)
print(n.lines.shape)
print(n.generators.shape)
print(n.loads.shape)
print(n.storage_units.shape)
buses_to_remove = []
for i in range(1, len(n.sub_networks)):
    buses_to_remove.extend(n.sub_networks.obj.iloc[i].buses().index.tolist())
print(buses_to_remove)
# Remove buses not in the first sub-network
n.mremove("Bus", buses_to_remove)
# Remove lines connected to the removed buses
lines_to_remove = []
for line in n.lines.index:
    if (n.lines.at[line, "bus0"] in buses_to_remove) or (n.lines.at[line, "bus1"] in buses_to_remove):
        lines_to_remove.append(line)
print(lines_to_remove)
n.mremove("Line", lines_to_remove)
# Remove generators connected to the removed buses
generators_to_remove = []
for generator in n.generators.index:
    if n.generators.at[generator, "bus"] in buses_to_remove:
        generators_to_remove.append(generator)
print(generators_to_remove)
n.mremove("Generator", generators_to_remove)
# Remove loads connected to the removed buses
loads_to_remove = []
for load in n.loads.index:
    if n.loads.at[load, "bus"] in buses_to_remove:
        loads_to_remove.append(load)
print(loads_to_remove)
n.mremove("Load", loads_to_remove)
# Remove storage_units connected to the removed buses
storage_units_to_remove = []
for storage_unit in n.storage_units.index:
    if n.storage_units.at[storage_unit, "bus"] in buses_to_remove:
        storage_units_to_remove.append(storage_unit)
print(storage_units_to_remove)
n.mremove("StorageUnit", storage_units_to_remove)
print(n.buses.shape)
print(n.lines.shape)
print(n.generators.shape)
print(n.loads.shape)
print(n.storage_units.shape)

(1096, 24)
(1564, 33)
(5446, 38)
(921, 6)
(2291, 32)
['36601', '36626', '36627', '36651', '36661', '36669', '37713', '37802']
['27', '28', '29', '30', '37']
['36601 onwind', '36601 solar existing', '36626 geothermal', '36626 onwind', '36626 solar', '36627 onwind', '36627 solar', '36651 hydro', '36651 onwind', '36651 solar', '36661 hydro', '36661 onwind', '36661 solar', '36669 hydro', '36669 onwind', '36669 solar', '37713 onwind', '37713 solar', '37802 onwind', '37802 solar existing', '36601 solar', '37802 solar', '36601 nuclear_2030', '36626 nuclear_2030', '36627 nuclear_2030', '36651 nuclear_2030', '36661 nuclear_2030', '36669 nuclear_2030', '37713 nuclear_2030', '37802 nuclear_2030', '36601 CCGT-95CCS_2030', '36626 CCGT-95CCS_2030', '36627 CCGT-95CCS_2030', '36651 CCGT-95CCS_2030', '36661 CCGT-95CCS_2030', '36669 CCGT-95CCS_2030', '37713 CCGT-95CCS_2030', '37802 CCGT-95CCS_2030', '36601 offwind_floating_2030', '36626 offwind_floating_2030', '36627 offwind_floating_2030', '36651 offwi

In [16]:
start_time = time.time()
n.lpf(snapshots=n.snapshots[:1])
print("Power flow calculation took %s seconds ---" % (time.time() - start_time))

INFO:pypsa.pf:Performing linear load-flow on AC sub-network SubNetwork 0 for snapshot(s) MultiIndex([(2030, '2030-01-01 00:00:00')],
           name='snapshot')


Power flow calculation took 48.670636892318726 seconds ---


In [17]:
n.loads_t.p_set.head(3)

Load                         36595 AC   36602 AC   36607 AC   36608 AC  \
period timestep                                                          
2030   2030-01-01 00:00:00  13.221867  13.941033  25.340700  22.133633   
       2030-01-01 03:00:00  13.366100  14.093133  25.617167  22.375133   
       2030-01-01 06:00:00  11.206767  11.816300  21.478633  18.760300   

Load                         36609 AC   36610 AC   36611 AC   36612 AC  \
period timestep                                                          
2030   2030-01-01 00:00:00  45.244000  28.397133  30.491433  30.491433   
       2030-01-01 03:00:00  45.737600  28.706933  30.824133  30.824133   
       2030-01-01 06:00:00  38.348467  24.069233  25.844367  25.844367   

Load                        36613 AC  36614 AC  ...  37719 AC  37722 AC  \
period timestep                                 ...                       
2030   2030-01-01 00:00:00  8.163433  9.839867  ...  0.194367  6.812567   
       2030-01-01 03:00:00  8.252500  9.947200  ...  0.196500  6.886933   
       2030-01-01 06:00:00  6.919267  8.340200  ...  0.164733  5.774300   

Load                        37723 AC  37724 AC  37732 AC  37733 AC   37734 AC  \
period timestep                                                                 
2030   2030-01-01 00:00:00  0.077767  0.753167  3.600700  1.700700  12.036200   
       2030-01-01 03:00:00  0.078600  0.761400  3.639933  1.719267  12.167567   
       2030-01-01 06:00:00  0.065900  0.638400  3.051900  1.441533  10.201833   

Load                        37737 AC   37738 AC  37753 AC  
period timestep                                            
2030   2030-01-01 00:00:00  3.445133  11.462867  0.204067  
       2030-01-01 03:00:00  3.482733  11.587900  0.206333  
       2030-01-01 06:00:00  2.920100   9.715867  0.173000  

[3 rows x 916 columns]

In [18]:
demand_p_summary = pd.DataFrame(n.loads_t.p_set).rename(columns=n.loads.bus).iloc[0].groupby(level=0).sum()
supply_p_summary = pd.DataFrame(n.loads_t.p).rename(columns=n.loads.bus).iloc[0].groupby(level=0).sum()
df = pd.DataFrame({"demand": demand_p_summary, "supply": supply_p_summary})
display(df.head())
(df.demand - df.supply).unique()

,demand,supply
Load,,
36595,13.221867,13.221867
36602,13.941033,13.941033
36607,25.340700,25.340700
36608,22.133633,22.133633
36609,45.244000,45.244000


array([0.])

In [19]:
demand_q_summary = pd.DataFrame(n.loads_t.q_set).rename(columns=n.loads.bus).iloc[0].groupby(level=0).sum()
supply_q_summary = pd.DataFrame(n.loads_t.q).rename(columns=n.loads.bus).iloc[0].groupby(level=0).sum()
df = pd.DataFrame({"demand": demand_q_summary, "supply": supply_q_summary})
display(df.head())
(df.demand - df.supply).unique()

,demand,supply
Load,,


array([], dtype=float64)

In [20]:
buses_gis = n.buses.copy()
buses_gis["geometry"] = [shapely.geometry.Point(xy) for xy in zip(buses_gis["x"], buses_gis["y"])]
gdf_buses = gpd.GeoDataFrame(buses_gis, geometry="geometry", crs="EPSG:4326")  # WGS84 lat/lon
gdf_buses.to_file(f"../results/SoCal/western/buses_cleaned.geojson", driver="GeoJSON")

lines = n.lines.copy()
bus_coords = n.buses[["x", "y"]]
lines["x0"] = lines["bus0"].map(bus_coords["x"])
lines["y0"] = lines["bus0"].map(bus_coords["y"])
lines["x1"] = lines["bus1"].map(bus_coords["x"])
lines["y1"] = lines["bus1"].map(bus_coords["y"])
lines["geometry"] = [
    shapely.geometry.LineString([(x0, y0), (x1, y1)])
    for x0, y0, x1, y1 in zip(lines["x0"], lines["y0"], lines["x1"], lines["y1"])
]
gdf_lines = gpd.GeoDataFrame(lines, geometry="geometry", crs="EPSG:4326")
gdf_lines.to_file(f"../results/SoCal/western/lines_cleaned.geojson", driver="GeoJSON")

generators = n.generators.copy()
generators["x"] = generators["bus"].map(bus_coords["x"])
generators["y"] = generators["bus"].map(bus_coords["y"])
generators["geometry"] = [shapely.geometry.Point(xy) for xy in zip(generators["x"], generators["y"])]
gdf_generators = gpd.GeoDataFrame(generators, geometry="geometry", crs="EPSG:4326")
gdf_generators.to_file(f"../results/SoCal/western/generators_cleaned.geojson", driver="GeoJSON")

INFO:pyogrio._io:Created 1,088 records


INFO:pyogrio._io:Created 1,559 records
INFO:pyogrio._io:Created 5,400 records


In [26]:
bus_to_lines = {}
for line in tqdm(n.lines.index):
    bus0 = n.lines.at[line, "bus0"]
    bus1 = n.lines.at[line, "bus1"]
    if bus0 not in bus_to_lines:
        bus_to_lines[bus0] = set()
    if bus1 not in bus_to_lines:
        bus_to_lines[bus1] = set()
    bus_to_lines[bus0].add(line)
    bus_to_lines[bus1].add(line)
for bus, lines in bus_to_lines.items():
    bus_to_lines[bus] = list(lines)

100%|██████████| 1559/1559 [00:00<00:00, 72315.59it/s]


In [28]:
import json
with open("../resources/SoCal/western/elec_s_call_ec_lv1.0_REM-3h_E_cleaned_bus_to_lines.json", "w") as f:
    json.dump(bus_to_lines, f)

In [31]:
n.lines.s_nom

Line
0       198.83
1       216.43
10      241.05
100     219.30
1000    238.01
         ...  
995     248.21
996     206.29
997     208.79
998     191.44
999     206.25
Name: s_nom, Length: 1559, dtype: float64

In [30]:
n.lines.columns

Index(['bus0', 'bus1', 'r', 'x', 'b', 's_nom', 'v_nom', 'interconnect',
       'carrier', 'underwater_fraction', 'length', 'capital_cost',
       'num_parallel', 'x_pu_eff', 'r_pu_eff', 's_nom_max', 's_max_pu', 'type',
       'g', 's_nom_mod', 's_nom_extendable', 's_nom_min', 'build_year',
       'lifetime', 'terrain_factor', 'v_ang_min', 'v_ang_max', 'sub_network',
       'x_pu', 'r_pu', 'g_pu', 'b_pu', 's_nom_opt'],
      dtype='object')

In [21]:
output_network = "../resources/SoCal/western/elec_s_call_ec_lv1.0_REM-3h_E_cleaned.nc"
n.export_to_netcdf(output_network)

INFO:pypsa.io:Exported network 'elec_s_call_ec_lv1.0_REM-3h_E_cleaned.nc' contains: buses, global_constraints, storage_units, generators, carriers, lines, loads


<xarray.Dataset> Size: 213MB
Dimensions:                               (snapshots: 2920,
                                           investment_periods: 1,
                                           buses_i: 1088, buses_t_p_i: 916,
                                           buses_t_v_ang_i: 1087,
                                           global_constraints_i: 1,
                                           storage_units_i: 2275,
                                           ...
                                           generators_t_p_max_pu_i: 2075,
                                           generators_t_p_i: 1, carriers_i: 19,
                                           lines_i: 1559, lines_t_p0_i: 1557,
                                           lines_t_p1_i: 1557, loads_i: 916,
                                           loads_t_p_set_i: 916,
                                           loads_t_p_i: 916)
Coordinates: (12/17)
  * snapshots                             (snapshots) int64 23kB 0 1 ... 2919
  * investment_periods                    (investment_periods) int64 8B 2030
  * buses_i                               (buses_i) object 9kB '36595' ... '3...
  * buses_t_p_i                           (buses_t_p_i) object 7kB '36595' .....
  * buses_t_v_ang_i                       (buses_t_v_ang_i) object 9kB '36602...
  * global_constraints_i                  (global_constraints_i) object 8B 'l...
    ...                                    ...
  * lines_i                               (lines_i) object 12kB '0' ... '999'
  * lines_t_p0_i                          (lines_t_p0_i) object 12kB '0' ... ...
  * lines_t_p1_i                          (lines_t_p1_i) object 12kB '0' ... ...
  * loads_i                               (loads_i) object 7kB '36595 AC' ......
  * loads_t_p_set_i                       (loads_t_p_set_i) object 7kB '36595...
  * loads_t_p_i                           (loads_t_p_i) object 7kB '36595 AC'...
Data variables: (12/93)
    snapshots_period                      (snapshots) int32 12kB 2030 ... 2030
    snapshots_timestep                    (snapshots) datetime64[ns] 23kB 203...
    snapshots_objective                   (snapshots) float64 23kB 3.0 ... 3.0
    snapshots_generators                  (snapshots) float64 23kB 3.0 ... 3.0
    snapshots_stores                      (snapshots) float64 23kB 3.0 ... 3.0
    investment_periods_objective          (investment_periods) float64 8B 1.0
    ...                                    ...
    lines_t_p0                            (snapshots, lines_t_p0_i) float64 36MB ...
    lines_t_p1                            (snapshots, lines_t_p1_i) float64 36MB ...
    loads_bus                             (loads_i) object 7kB '36595' ... '3...
    loads_carrier                         (loads_i) object 7kB 'AC' ... 'AC'
    loads_t_p_set                         (snapshots, loads_t_p_set_i) float64 21MB ...
    loads_t_p                             (snapshots, loads_t_p_i) float64 21MB ...
Attributes:
    network_name:           
    network_pypsa_version:  0.30.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"run": {"name": "SoCal", "disable_progressbar": ...